In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
import warnings
warnings.filterwarnings('ignore')

---
## Step 1: Load and Explore the Dataset

In [2]:
df = pd.read_csv("../Dataset/SAVSNET.csv", index_col=0, encoding='latin-1') 

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (4415, 5)
Columns: ['SAVSNET_consult_id', 'Narrative', 'SAVSNET MPC', 'Consult_date', 'Species']

First few rows:


,SAVSNET_consult_id,Narrative,SAVSNET MPC,Consult_date,Species
0,1111025,"""Booster: hc satis. O no concerns. Ce NAD. Una...",vaccination,2016-01-27 08:40:27,dog
1,6761083,some tartar care w/ weight! reduce calories by...,vaccination,2019-10-03 11:24:06,dog
2,3938657,"""Came in for 6 month HPC check. BAR Hr=100 mm=...",other_healthy,2017-11-29 17:47:49,unknown
3,6081319,lame RF 5 days possibly getting better - had l...,trauma,2019-05-09 09:24:07,dog
4,4162251,"""H: Ate some plastic approx a week ago. This h...",other_unwell,2018-01-30 16:55:57,dog


In [3]:
print(f"Total records: {len(df)}")
print(f"\nSpecies distribution:")
print(df['Species'].value_counts())
print(f"\nSAVSNET MPC (Main Presenting Complaint) categories:")
print(df['SAVSNET MPC'].value_counts())
print(f"\nDate range: {df['Consult_date'].min()} to {df['Consult_date'].max()}") 
print(f"\nMissing narratives: {df['Narrative'].isna().sum()}")
print(f"\nAverage narrative length: {df['Narrative'].str.len().mean():.0f} characters")

Total records: 4415

Species distribution:
Species
dog               2907
cat               1169
unknown            237
rabbit              72
guinea pig          13
hamster              4
rat                  4
ferret               3
budgerigar           2
bearded dragon       1
chinchilla           1
canary               1
cockatiel            1
Name: count, dtype: int64

SAVSNET MPC (Main Presenting Complaint) categories:
SAVSNET MPC
vaccination       1440
other_healthy     1076
other_unwell       817
post_op            380
trauma             203
pruritus           199
gastroenteric      139
tumour              70
respiratory         53
kidney_disease      23
Name: count, dtype: int64

Date range: 2014-03-24 15:33:03 to 2019-10-22 19:19:56

Missing narratives: 0

Average narrative length: 301 characters


In [4]:
df_dogs = df[df['Species'] == 'dog'].copy()
print(f"Dog records: {len(df_dogs)} out of {len(df)} total ({len(df_dogs)/len(df)*100:.1f}%)")

df_dogs = df_dogs.dropna(subset=['Narrative'])
df_dogs = df_dogs[df_dogs['Narrative'].str.len() > 10]  
df_dogs['Consult_date'] = pd.to_datetime(df_dogs['Consult_date'])
df_dogs['year'] = df_dogs['Consult_date'].dt.year
df_dogs['month'] = df_dogs['Consult_date'].dt.month
df_dogs['year_month'] = df_dogs['Consult_date'].dt.to_period('M')

docs = df_dogs['Narrative'].tolist()
print(f"\nReady for BERTopic: {len(docs)} documents")

Dog records: 2907 out of 4415 total (65.8%)

Ready for BERTopic: 2771 documents


---
## Step 2: Run BERTopic Pipeline

In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

umap_model = UMAP(
    n_components=5,      
    n_neighbors=15,      
    min_dist=0.0,         
    metric='cosine',      
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,   
    min_samples=5,         
    metric='euclidean',
    prediction_data=True   
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),    
    min_df=2              
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,  
    verbose=True
)

print("BERTopic model configured!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BERTopic model configured!


In [6]:
topics, probs = topic_model.fit_transform(docs)
df_dogs['topic'] = topics

if -1 in topics:
    total_topics = len(set(topics)) - 1
else:
    total_topics = len(set(topics))

outlier_count = 0
for t in topics:
    if t == -1:
        outlier_count += 1

documents_with_topics = 0
for t in topics:
    if t != -1:
        documents_with_topics += 1

outlier_percentage = (outlier_count / len(topics)) * 100

print("\nBERTopic Results:")
print(f"  Total topics found: {total_topics}")
print(f"  Outliers (topic -1): {outlier_count} ({outlier_percentage:.1f}%)")
print(f"  Documents with topics: {documents_with_topics}")

2026-05-13 14:57:00,052 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/87 [00:00<?, ?it/s]

2026-05-13 14:57:27,447 - BERTopic - Embedding - Completed ✓
2026-05-13 14:57:27,449 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-13 14:57:46,751 - BERTopic - Dimensionality - Completed ✓
2026-05-13 14:57:46,752 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-13 14:57:47,064 - BERTopic - Cluster - Completed ✓
2026-05-13 14:57:47,070 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-13 14:57:47,380 - BERTopic - Representation - Completed ✓



BERTopic Results:
  Total topics found: 47
  Outliers (topic -1): 702 (25.3%)
  Documents with topics: 2069


In [7]:
topic_info = topic_model.get_topic_info()                                       
                                                                                  
topic_display = topic_info[['Topic', 'Count', 'Name']].head(20).copy()
topic_display['Name'] = topic_display['Name'].str[:40]

print(f"{'Topic':<8} {'Count':<8} {'Name'}")
print("-" * 60)

for _, row in topic_display.iterrows():
    print(f"{int(row['Topic']):<8} {int(row['Count']):<8} {row['Name']}")

Topic    Count    Name
------------------------------------------------------------
-1       702      -1_ok_kc_l4_fine
0        247      0_ear_ears_canal_head
1        204      1_wound_poc_healed_bandage
2        157      2_pain_lame_lameness_rest
3        153      3_food_blood_diet_faeces
4        135      4_eye_ulcer_left eye_eyes
5        108      5_booster_kc_l4_nad
6        104      6_skin_fleas_tail_flea
7        78       7_booster_kc_check check_abnormalities d
8        73       8_lump_fna_mass_cyst
9        46       9_cough_coughing_chest_tracheal
10       44       10_urine_lab_bladder_sample
11       41       11_vaccination_flea_2nd vaccination_pupp
12       40       12_dental_teeth_gingivitis_mouth
13       40       13_nails_clip_clipped_nail
14       36       14_apoquel_skin_dose_months
15       35       15_claw_nail_dew claw_dew
16       33       16_vaccination_vaccination l4_l4_concern
17       33       17_vacc_flea_vac_2nd
18       32       18_appointment_appointment week

---
## Step 3: Topic Coherence (c_v)

In [8]:
def calculate_coherence(topic_model, docs):
    topic_words = []

    for topic_id in sorted(topic_model.get_topics().keys()):
        if topic_id == -1:
            continue

        words = []
        for word, score in topic_model.get_topic(topic_id):
            words.append(word)

        topic_words.append(words)

    texts = []

    for doc in docs:
        tokenized_doc = doc.lower().split()
        texts.append(tokenized_doc)

    dictionary = Dictionary(texts)

    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )

    overall = coherence_model.get_coherence()
    per_topic = coherence_model.get_coherence_per_topic()

    return overall, per_topic

print("Calculating coherence scores..")
coherence_score, per_topic_coherence = calculate_coherence(topic_model, docs)

print(f"\n{'='*40}")
print("COHERENCE RESULTS")
print(f"{'='*40}")
print(f"Average c_v coherence: {coherence_score:.4f}")
print("")

good = 0
acceptable = 0
poor = 0

for coherence in per_topic_coherence:
    if coherence >= 0.7:
        good += 1
    elif coherence >= 0.5:
        acceptable += 1
    else:
        poor += 1

total = len(per_topic_coherence)

good_percentage = (good / total) * 100
acceptable_percentage = (acceptable / total) * 100
poor_percentage = (poor / total) * 100

print(f"Good (>= 0.7):         {good}/{total} ({good_percentage:.1f}%)")
print(f"Acceptable (0.5-0.7): {acceptable}/{total} ({acceptable_percentage:.1f}%)")
print(f"Poor (< 0.5):         {poor}/{total} ({poor_percentage:.1f}%)")

Calculating coherence scores..

COHERENCE RESULTS
Average c_v coherence: 0.4838

Good (>= 0.7):         6/47 (12.8%)
Acceptable (0.5-0.7): 16/47 (34.0%)
Poor (< 0.5):         25/47 (53.2%)


In [9]:
if probs.ndim == 2:
    max_probs = probs.max(axis=1)
else:
    max_probs = probs

print(f"TOPIC ASSIGNMENT PROBABILITY")
print(f"{'='*30}")
print(f"Median max probability: {np.median(max_probs):.4f}")
print(f"Mean max probability:   {np.mean(max_probs):.4f}")
print(f"Docs with > 0.5:        {np.sum(max_probs > 0.5)/len(max_probs)*100:.1f}%")
print(f"")

TOPIC ASSIGNMENT PROBABILITY
Median max probability: 0.2321
Mean max probability:   0.4073
Docs with > 0.5:        32.3%



---
## Step 4: Odds Ratio (OR), Standard Error (SE), and 95% Confidence Interval (CI)
- **OR** = (a × d) / (b × c) with +1 continuity correction
- **SE(log OR)** = sqrt(1/a + 1/b + 1/c + 1/d)
- **CI** = exp(log(OR) ± 1.96 × SE)

Where:
- a = count of topic in group of interest
- b = count of topic in reference group
- c = count of other topics in group of interest
- d = count of other topics in reference group

In [10]:
def calculate_or_se_ci(df, topic_id, group_col, group_value, ref_value):
    group = df[df[group_col] == group_value]
    ref = df[df[group_col] == ref_value]
    
    a = len(group[group['topic'] == topic_id]) + 1   
    b = len(ref[ref['topic'] == topic_id]) + 1      
    c = len(group[group['topic'] != topic_id]) + 1  
    d = len(ref[ref['topic'] != topic_id]) + 1       
    
    OR = (a * d) / (b * c)
    
    SE = np.sqrt(1/a + 1/b + 1/c + 1/d)
    
    log_OR = np.log(OR)
    CI_lower = np.exp(log_OR - 1.96 * SE)
    CI_upper = np.exp(log_OR + 1.96 * SE)
    
    significant = (CI_lower > 1.0) or (CI_upper < 1.0)
    
    return {
        'group': group_value,
        'topic': topic_id,
        'OR': round(OR, 4),
        'SE': round(SE, 4),
        'CI_lower': round(CI_lower, 4),
        'CI_upper': round(CI_upper, 4),
        'log_OR': round(log_OR, 4),
        'significant': significant
    }

print("OR/SE/CI function ready!")

OR/SE/CI function ready!


---
## Step 5: Breed Distribution (Forest Plot)

In [11]:
print("Available columns for grouping:")
print(df_dogs.columns.tolist())
print()

mpc_counts = df_dogs['SAVSNET MPC'].value_counts()
print("MPC category counts:")
print(mpc_counts)

Available columns for grouping:
['SAVSNET_consult_id', 'Narrative', 'SAVSNET MPC', 'Consult_date', 'Species', 'year', 'month', 'year_month', 'topic']

MPC category counts:
SAVSNET MPC
vaccination       896
other_healthy     660
other_unwell      494
post_op           249
pruritus          140
trauma            138
gastroenteric      97
tumour             50
respiratory        30
kidney_disease      7
Name: count, dtype: int64


In [12]:
reference_mpc = "vaccination"

top_topics = []

for topic in df_dogs['topic'].value_counts().index:
    if topic != -1:
        top_topics.append(topic)

top_topics = top_topics[:10]

print(f"Calculating OR for MPC categories (reference: {reference_mpc})")
print(f"Top {len(top_topics)} topics analyzed")
print()

or_results = []
mpc_categories = []

for mpc in df_dogs['SAVSNET MPC'].unique():
    if mpc != reference_mpc:
        mpc_categories.append(mpc)

for mpc in mpc_categories:
    for topic_id in top_topics:
        result = calculate_or_se_ci(df_dogs,topic_id,'SAVSNET MPC',mpc,reference_mpc)

        topic_words = []

        for word, score in topic_model.get_topic(topic_id)[:3]:
            topic_words.append(word)

        result['topic_name'] = '_'.join(topic_words)
        result['mpc'] = mpc

        or_results.append(result)

or_df = pd.DataFrame(or_results)

sig_results = or_df[or_df['significant'] == True]
sig_results = sig_results.sort_values('OR', ascending=False)

print(f"Significant associations found: {len(sig_results)}")
print()

if len(sig_results) > 0:
    display_df = sig_results[['mpc', 'topic_name', 'OR', 'CI_lower', 'CI_upper']].head(15).copy()
    
    print(f"{'MPC':<20} {'Topic Name':<25} {'OR':<8} {'CI Lower':<10} {'CI Upper'}")
    print("-" * 75)

    for _, row in display_df.iterrows():
        print(f"{row['mpc']:<20} {row['topic_name']:<25} {row['OR']:<8.2f} {row['CI_lower']:<10.2f} {row['CI_upper']:.2f}")

Calculating OR for MPC categories (reference: vaccination)
Top 10 topics analyzed

Significant associations found: 46

MPC                  Topic Name                OR       CI Lower   CI Upper
---------------------------------------------------------------------------
respiratory          cough_coughing_chest      576.00   121.85     2722.93
nan                  cough_coughing_chest      448.00   20.19      9941.94
gastroenteric        food_blood_diet           161.96   79.58      329.60
nan                  food_blood_diet           73.83    4.36       1250.87
tumour               lump_fna_mass             58.87    27.93      124.08
nan                  lump_fna_mass             58.87    3.51       986.07
nan                  wound_poc_healed          58.87    3.51       986.07
kidney_disease       cough_coughing_chest      56.00    4.60       681.80
nan                  eye_ulcer_left eye        55.12    3.30       920.77
nan                  skin_fleas_tail           48.89    2.94

In [13]:
def plot_forest(or_df, topic_id, title=""):
    subset = or_df[or_df['topic'] == topic_id].copy()
    subset = subset.sort_values('OR', ascending=True)

    if len(subset) == 0:
        print(f"No data for topic {topic_id}")
        return

    topic_words = [w for w, _ in topic_model.get_topic(topic_id)][:5]
    topic_label = ', '.join(topic_words)

    fig = go.Figure()

    for i, row in subset.iterrows():
        color = 'red' if row['significant'] else 'gray'
        fig.add_trace(go.Scatter(
            x=[row['CI_lower'], row['CI_upper']],
            y=[row['mpc'], row['mpc']],
            mode='lines',
            line=dict(color=color, width=2),
            showlegend=False
        ))

    fig.add_trace(go.Scatter(
        x=subset['OR'],
        y=subset['mpc'],
        mode='markers',
        marker=dict(size=10, color=['red' if s else 'gray' for s in subset['significant']], symbol='diamond'),
        name='Odds Ratio'
    ))

    fig.add_vline(x=1, line_dash="dash", line_color="black", opacity=0.5)

    fig.update_layout(
        title=f"Forest Plot - Topic: [{topic_label}]<br><sub>{title}</sub>",
        xaxis_title="Odds Ratio (log scale)",
        xaxis_type="log",
        yaxis_title="MPC Category",
        height=550,
        width=1000,
        margin=dict(l=250, r=100, t=120, b=80),
        showlegend=False
    )

    fig.update_yaxes(
        automargin=True,
        tickfont=dict(size=13),
        ticklabelstandoff=25
    )

    fig.show()

print("Red = significant (CI doesn't cross 1.0), Gray = not significant")
print()

for topic_id in top_topics[:3]:
    plot_forest(or_df, topic_id, "Reference: vaccination")

Red = significant (CI doesn't cross 1.0), Gray = not significant



---
## Step 6: Age Distribution
Different topics occur at different ages:
- Young dogs: vaccination, puppy checks
- Middle-aged: lumps/masses
- Older dogs: vestibular disease, euthanasia

In [14]:

print("Topic Distribution by MPC Category")
print()

topic_mpc = pd.crosstab(df_dogs['SAVSNET MPC'], df_dogs['topic'], normalize='index')

for mpc in df_dogs['SAVSNET MPC'].value_counts().index:
    row = topic_mpc.loc[mpc]
    if -1 in row.index:
        row = row.drop(-1)

    top_topic = row.idxmax()
    top_pct = row.max() * 100
    topic_data = topic_model.get_topic(top_topic)

    words = []

    for item in topic_data[:4]:
        word = item[0]
        words.append(word)
        
    print(f"  {mpc:15s} = Topic {top_topic} ({top_pct:.1f}%): {words}")

Topic Distribution by MPC Category

  vaccination     = Topic 5 (11.2%): ['booster', 'kc', 'l4', 'nad']
  other_healthy   = Topic 0 (10.6%): ['ear', 'ears', 'canal', 'head']
  other_unwell    = Topic 0 (13.4%): ['ear', 'ears', 'canal', 'head']
  post_op         = Topic 1 (43.8%): ['wound', 'poc', 'healed', 'bandage']
  pruritus        = Topic 0 (46.4%): ['ear', 'ears', 'canal', 'head']
  trauma          = Topic 2 (39.9%): ['pain', 'lame', 'lameness', 'rest']
  gastroenteric   = Topic 3 (69.1%): ['food', 'blood', 'diet', 'faeces']
  tumour          = Topic 8 (50.0%): ['lump', 'fna', 'mass', 'cyst']
  respiratory     = Topic 9 (56.7%): ['cough', 'coughing', 'chest', 'tracheal']
  kidney_disease  = Topic 10 (57.1%): ['urine', 'lab', 'bladder', 'sample']


---
## Step 7: Time Series Analysis
Analyzed topic proportions over time to detect:
- Disease outbreaks
- Seasonal patterns
- Long-term trends
- COVID lockdown effects

In [15]:

monthly = df_dogs.groupby(['year_month', 'topic']).size().reset_index(name='count')
monthly_total = df_dogs.groupby('year_month').size().reset_index(name='total')
monthly = monthly.merge(monthly_total, on='year_month')
monthly['proportion'] = monthly['count'] / monthly['total']
monthly['date'] = monthly['year_month'].dt.to_timestamp()

fig = go.Figure()
colors = px.colors.qualitative.Set1

i = 0

for topic_id in top_topics[:5]:
    topic_data = monthly[monthly['topic'] == topic_id]
    topic_data = topic_data.sort_values('date')

    if len(topic_data) == 0:
        continue

    max_proportion = topic_data['proportion'].max()

    if max_proportion <= 0:
        continue

    normalized = topic_data['proportion'] / max_proportion
    topic_words_data = topic_model.get_topic(topic_id)

    words = []

    for item in topic_words_data[:3]:
        word = item[0]
        words.append(word)

    joined_words = ', '.join(words)
    label = f"Topic {topic_id}: {joined_words}"
    color_index = i % len(colors)
    selected_color = colors[color_index]

    fig.add_trace(go.Scatter(
        x=topic_data['date'],
        y=normalized,
        mode='lines+markers',
        name=label,
        line=dict(color=selected_color),
        marker=dict(size=4)
    ))

    i = i + 1

fig.update_layout(
    title="Topic Proportions Over Time (Normalized to Peak)",
    xaxis_title="Date",
    yaxis_title="Normalized Proportion (1.0 = peak)",
    height=500,
    width=900,
    margin=dict(
        l=90,   
        r=50,   
        t=80,  
        b=150   
    ),

    legend=dict(
        x=0.5,          
        y=-0.3,
        xanchor='center',
        orientation='h'
    )
)

fig.show()

In [16]:
fig = go.Figure()

for i, topic_id in enumerate(top_topics[:5]):
    topic_data = monthly[monthly['topic'] == topic_id].sort_values('date')
    
    if len(topic_data) == 0:
        continue
    
    words = [w for w, _ in topic_model.get_topic(topic_id)][:3]
    label = f"Topic {topic_id}: {', '.join(words)}"
    
    fig.add_trace(go.Scatter(
        x=topic_data['date'],
        y=topic_data['proportion'] * 100,
        mode='lines+markers',
        name=label,
        line=dict(color=colors[i % len(colors)]),
        marker=dict(size=4)
    ))

fig.update_layout(
    title="Topic Proportions Over Time (Raw %)<br><sub>Percentage of consultations assigned to each topic per month</sub>",
    xaxis_title="Date",
    yaxis_title="Proportion (%)",
    height=500,
    width=900,
    legend=dict(x=0, y=-0.3, orientation='h'),
    margin=dict(
        l=90,   
        r=50,   
        t=80,  
        b=150   
    ),

)

fig.update_yaxes(
    automargin=True,
    tickfont=dict(size=12),
    ticklabelstandoff=10,
    title_standoff=20
)

fig.update_xaxes(
    automargin=True,
    tickfont=dict(size=12),
    ticklabelstandoff=10,
    title_standoff=20
)

fig.show()

---
## Step 8: Sex Distribution (OR)
Compared topic distributions between male and female dogs.

In [17]:

group_a = "pruritus"
group_b = "vaccination"

print(f"OR Comparison: '{group_a}' vs '{group_b}' (reference)")
print(f"{'='*90}")
print()

comparison_results = []

for topic_id in top_topics:
    result = calculate_or_se_ci(df_dogs, topic_id, 'SAVSNET MPC', group_a, group_b)
    topic_words_data = topic_model.get_topic(topic_id)

    words = []

    for item in topic_words_data[:4]:
        word = item[0]
        words.append(word)

    joined_words = ', '.join(words)
    result['topic_words'] = joined_words
    comparison_results.append(result)

comp_df = pd.DataFrame(comparison_results)
comp_df = comp_df.sort_values('OR', ascending=False)
print(f"{'Topic':<50} {'OR':>8} {'CI_lower':>10} {'CI_upper':>10} {'Sig?':>6}")
print("-" * 90)

for index, row in comp_df.iterrows():
    if row['significant'] == True:
        sig = "YES"
    else:
        sig = "no"

    print(
        f"{row['topic_words']:<50} "
        f"{row['OR']:>8.3f} "
        f"{row['CI_lower']:>10.3f} "
        f"{row['CI_upper']:>10.3f} "
        f"{sig:>6}")

OR Comparison: 'pruritus' vs 'vaccination' (reference)

Topic                                                    OR   CI_lower   CI_upper   Sig?
------------------------------------------------------------------------------------------
ear, ears, canal, head                               22.763     14.097     36.757    YES
skin, fleas, tail, flea                              21.235     11.793     38.236    YES
cough, coughing, chest, tracheal                      3.177      0.286     35.272     no
eye, ulcer, left eye, eyes                            2.012      0.725      5.580     no
wound, poc, healed, bandage                           1.270      0.363      4.445     no
lump, fna, mass, cyst                                 0.841      0.190      3.717     no
food, blood, diet, faeces                             0.524      0.068      4.059     no
pain, lame, lameness, rest                            0.347      0.046      2.618     no
booster, kc, check check, abnormalities detected    

In [18]:

comp_df_sorted = comp_df.sort_values('OR')

y_labels = []

for index, row in comp_df_sorted.iterrows():
    topic_number = int(row['topic'])
    words = row['topic_words'].split(',')
    short_keywords = []

    for word in words[:3]:
        cleaned_word = word.strip()
        short_keywords.append(cleaned_word)

    short_text = ", ".join(short_keywords)
    label = f"T{topic_number}: {short_text}"
    y_labels.append(label)

marker_colors = []

for significant_value in comp_df_sorted['significant']:
    if significant_value == True:
        marker_colors.append('red')
    else:
        marker_colors.append('steelblue')

error_array = []

for index, row in comp_df_sorted.iterrows():
    upper_error = row['CI_upper'] - row['OR']
    error_array.append(upper_error)

error_arrayminus = []

for index, row in comp_df_sorted.iterrows():
    lower_error = row['OR'] - row['CI_lower']
    error_arrayminus.append(lower_error)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=comp_df_sorted['OR'],
        y=y_labels,
        mode='markers',
        marker=dict(
            size=12,
            color=marker_colors,
            symbol='diamond'
        ),

        error_x=dict(
            type='data',
            symmetric=False,
            array=error_array,
            arrayminus=error_arrayminus,
            color='gray',
            thickness=1.5
        ),

        name='OR with 95% CI'
    )
)

fig.add_vline(
    x=1,
    line_dash="dash",
    line_color="black",
    opacity=0.5
)

fig.update_layout(
    title=f"Odds Ratio: {group_a} vs {group_b}",
    xaxis_title="Odds Ratio (log scale)",
    xaxis_type="log",
    height=600,
    width=1000,
    margin=dict(
        l=300,
        r=90,
        t=70,
        b=80
    ),
    showlegend=False
)

fig.update_yaxes(
    automargin=True,
    tickfont=dict(size=14),
    ticklabelstandoff=25
)

fig.show()

---
## Step 9: Topic Visualization (BERTopic built-in)
BERTopic provides built-in visualization methods

In [19]:
fig = topic_model.visualize_barchart(top_n_topics=8, n_words=10)
fig.update_layout(title="Top Topic Keywords (c-TF-IDF weights)")

fig = topic_model.visualize_barchart(
    top_n_topics=8,
    n_words=10
)

fig.update_layout(
    title="Top Topic Keywords (c-TF-IDF weights)",
    width=1200,
    height=800,
    margin=dict(t=80, b=50, l=50, r=50),
    font=dict(size=12)
)

fig.update_xaxes(
    tickangle=0,      
    title_font=dict(size=10),
    tickfont=dict(size=10)
)

fig.update_yaxes(
    title_font=dict(size=10),
    tickfont=dict(size=10)
)

fig.show()

In [20]:

timestamps = df_dogs['Consult_date'].tolist()

topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps,
    nr_bins=20
)

fig = topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=7 
)


topic_info = topic_model.get_topics()

new_labels = {}
for topic_id in topic_info:
    if topic_id == -1:
        continue

    words = [word for word, _ in topic_info[topic_id][:4]]
    label = f"T{topic_id}: " + ", ".join(words)
    new_labels[topic_id] = label

for trace in fig.data:
    try:
        topic_id = int(trace.name.split("_")[0])
        if topic_id in new_labels:
            trace.name = new_labels[topic_id]
    except:
        pass


fig.update_layout(
    title="Topics Over Time (BERTopic - Clean View)", 
    width=1200,
    height=600,
     margin=dict(
        l=90,   
        r=50,   
        t=80,  
        b=50  
    ),


    legend=dict(
        title="Topics",
        font=dict(size=10)
    )
)

fig.update_xaxes(title="Time")
fig.update_yaxes(title="Frequency")

fig.show()

20it [00:01, 18.03it/s]


In [21]:
mpc_counts = df_dogs['SAVSNET MPC'].value_counts()

valid_mpcs = []
for mpc, count in mpc_counts.items():
    if count >= 20:
        valid_mpcs.append(mpc)

mask = df_dogs['SAVSNET MPC'].isin(valid_mpcs)

docs_filtered = []
classes_filtered = []

for i in df_dogs[mask].index:
    docs_filtered.append(df_dogs.loc[i, 'Narrative'])
    classes_filtered.append(df_dogs.loc[i, 'SAVSNET MPC'])

print("Using", len(valid_mpcs), "MPC categories with >= 20 docs each")
print("Documents used:", len(docs_filtered), "/", len(docs))

try:
    topics_per_class = topic_model.topics_per_class(
        docs_filtered,
        classes=classes_filtered
    )

    fig = topic_model.visualize_topics_per_class(
        topics_per_class,
        top_n_topics=8
    )

    fig.update_layout(
        title="Topics per MPC Category",
        height=600
    )

    fig.show()

except Exception as e:
    print("\nAlternative: showing topic counts per MPC manually\n")

    cross = pd.crosstab(df_dogs['SAVSNET MPC'], df_dogs['topic'])

    if -1 in cross.columns:
        cross = cross.drop(columns=[-1])

    topic_sums = cross.sum()
    top_8 = topic_sums.nlargest(8).index.tolist()

    cross = cross.reindex(columns=top_8, fill_value=0)

    cross = cross.sort_index()
    cross = cross.sort_index(axis=1)

    cross = cross.fillna(0)
    cross = cross.astype(int)

print("Topic Counts per MPC Category")
print()

col_width = 10 

header_row = f"{'MPC':<15}"  
for col in cross.columns:
    header_row += f"{col:>{col_width}}"
print(header_row)
print("-" * len(header_row))

for idx in cross.index:
    row_str = f"{idx:<15}" 
    for col in cross.columns:
        val = cross.loc[idx, col]
        row_str += f"{val:>{col_width}}"
    print(row_str)

Using 9 MPC categories with >= 20 docs each
Documents used: 2754 / 2771

Alternative: showing topic counts per MPC manually

Topic Counts per MPC Category

MPC                     0         1         2         3         4         5         6         7
-----------------------------------------------------------------------------------------------
gastroenteric           1         1         1        67         0         0         0         0
kidney_disease          1         0         0         0         0         0         0         0
other_healthy          70        24        31        31        32         4        21        16
other_unwell           66        27        41        37        60         4        14         4
post_op                 8       109         9         6        12         0         5         5
pruritus               65         2         0         0         4         0        42         0
respiratory             0         0         0         1         0         0 

---
## Step 10: Complete OR/SE/CI Table for All Topics
Generate a comprehensive table like the paper would use for all analyses.

In [22]:
print("COMPLETE OR/SE/CI ANALYSIS")
print(f"Reference group: {reference_mpc}")
print("=" * 85)

all_or = []

for mpc in mpc_categories:
    for topic_id in top_topics:
        result = calculate_or_se_ci(
            df_dogs,
            topic_id,
            'SAVSNET MPC',
            mpc,
            reference_mpc
        )

        topic_words_data = topic_model.get_topic(topic_id)

        words = []

        for item in topic_words_data[:3]:
            word = item[0]
            words.append(word)

        joined_words = ', '.join(words)
        result['topic_name'] = joined_words
        all_or.append(result)

full_or_df = pd.DataFrame(all_or)

print(f"\nTotal comparisons: {len(full_or_df)}")
significant_count = full_or_df['significant'].sum()
significant_percentage = (
    significant_count / len(full_or_df)
) * 100

print(
    f"Significant results: "
    f"{significant_count} "
    f"({significant_percentage:.1f}%)"
)

or_min = full_or_df['OR'].min()
or_max = full_or_df['OR'].max()
print(f"\nOR range: {or_min:.4f} to {or_max:.4f}")
mean_se = full_or_df['SE'].mean()
print(f"Mean SE: {mean_se:.4f}")

print(f"\nTop 10 Significant Associations (highest OR):")
print("-" * 85)

top_sig = full_or_df[full_or_df['significant'] == True]
top_sig = top_sig.sort_values('OR', ascending=False)
top_sig = top_sig.head(10)

if len(top_sig) > 0:
    for index, row in top_sig.iterrows():
        grp = str(row['group'])
        tname = str(row['topic_name'])

        print(
            f"  {grp:15s} | "
            f"Topic: {tname:25s} | "
            f"OR={row['OR']:.3f} "
            f"[{row['CI_lower']:.3f}, {row['CI_upper']:.3f}]"
        )

else:
    print("  No significant results (dataset may be too small)")

print(f"\nTop 10 Significant Associations (lowest OR - protective):")
print("-" * 85)

bot_sig = full_or_df[full_or_df['significant'] == True]
bot_sig = bot_sig.sort_values('OR', ascending=True)

bot_sig = bot_sig.head(10)

if len(bot_sig) > 0:
    for index, row in bot_sig.iterrows():
        grp = str(row['group'])
        tname = str(row['topic_name'])
        print(
            f"  {grp:15s} | "
            f"Topic: {tname:25s} | "
            f"OR={row['OR']:.3f} "
            f"[{row['CI_lower']:.3f}, {row['CI_upper']:.3f}]"
        )

else:
    print("  No significant results (dataset may be too small)")

COMPLETE OR/SE/CI ANALYSIS
Reference group: vaccination

Total comparisons: 100
Significant results: 46 (46.0%)

OR range: 0.0316 to 576.0000
Mean SE: 0.8002

Top 10 Significant Associations (highest OR):
-------------------------------------------------------------------------------------
  respiratory     | Topic: cough, coughing, chest    | OR=576.000 [121.845, 2722.925]
  nan             | Topic: cough, coughing, chest    | OR=448.000 [20.188, 9941.936]
  gastroenteric   | Topic: food, blood, diet         | OR=161.957 [79.581, 329.601]
  nan             | Topic: food, blood, diet         | OR=73.833 [4.358, 1250.868]
  tumour          | Topic: lump, fna, mass           | OR=58.867 [27.928, 124.077]
  nan             | Topic: lump, fna, mass           | OR=58.867 [3.514, 986.067]
  nan             | Topic: wound, poc, healed        | OR=58.867 [3.514, 986.067]
  kidney_disease  | Topic: cough, coughing, chest    | OR=56.000 [4.600, 681.795]
  nan             | Topic: eye, ulcer, lef

In [23]:
print("=" * 50)
print("Summary: BERTopic Analysis of SAVSNET Data")
print("=" * 50)

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(1 for t in topics if t == -1)

print(f"""
DATASET:
  Total records:     {len(df)}
  Dog records used:  {len(df_dogs)}
  Date range:        {df_dogs['Consult_date'].min().strftime('%Y-%m-%d')} to {df_dogs['Consult_date'].max().strftime('%Y-%m-%d')}
  MPC categories:    {df_dogs['SAVSNET MPC'].nunique()}

BERTOPIC RESULTS:
  Topics found:      {n_topics}
  Outliers:          {n_outliers} ({n_outliers/len(topics)*100:.1f}%)
  Coherence (c_v):   {coherence_score:.4f}
  Median probability: {np.median(max_probs):.4f}
  Mean probability:   {np.mean(max_probs):.4f}

OR/SE/CI ANALYSIS:
  Total comparisons: {len(full_or_df)}
  Significant:       {full_or_df['significant'].sum()} ({full_or_df['significant'].sum()/len(full_or_df)*100:.1f}%)
""")

print("=" * 50)

Summary: BERTopic Analysis of SAVSNET Data

DATASET:
  Total records:     4415
  Dog records used:  2771
  Date range:        2014-03-24 to 2019-10-22
  MPC categories:    10

BERTOPIC RESULTS:
  Topics found:      47
  Outliers:          702 (25.3%)
  Coherence (c_v):   0.4838
  Median probability: 0.2321
  Mean probability:   0.4073

OR/SE/CI ANALYSIS:
  Total comparisons: 100
  Significant:       46 (46.0%)

